Change cell bellow to point to your fork !

<a href="https://colab.research.google.com/github/PabloRR100/intro_deep_learning/blob/main/hackathon/notebook/template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Similarity Engine

Build a similarity engine using pre-trained compute vision models

In [1]:
!unzip "/content/data.zip" -d /content/

Archive:  /content/data.zip
   creating: /content/data/
   creating: /content/data/database/
   creating: /content/data/database/bulbasaur/
  inflating: /content/data/database/bulbasaur/1.jpg  
  inflating: /content/data/database/bulbasaur/2.jpg  
  inflating: /content/data/database/bulbasaur/bulbasaur.png  
  inflating: /content/data/database/bulbasaur/bulbasaur_1.jpeg  
   creating: /content/data/database/charmander/
  inflating: /content/data/database/charmander/1.jpg  
  inflating: /content/data/database/charmander/2.jpg  
  inflating: /content/data/database/charmander/3.jpg  
  inflating: /content/data/database/charmander/charmander.jpeg  
   creating: /content/data/database/squirtle/
  inflating: /content/data/database/squirtle/1.jpg  
  inflating: /content/data/database/squirtle/2.jpg  
  inflating: /content/data/database/squirtle/squirtle.png  
  inflating: /content/data/database/squirtle/squirtle_1.jpeg  
   creating: /content/data/embeddings/
  inflating: /content/data/embedd

In [2]:
!ls "/content/data/database"

bulbasaur  charmander  squirtle


In [79]:
import os
import torch
import pickle
from urllib.request import urlopen
from PIL import Image

import timm

import transformers
from transformers import ViTForImageClassification, ViTFeatureExtractor, ViTModel, ViTImageProcessor
from transformers import MobileViTImageProcessor, MobileViTForImageClassification, MobileViTModel

import warnings
warnings.filterwarnings('ignore')

import logging
logging.disable(logging.WARNING)

from collections import defaultdict

In [4]:
# --- 1. Load Pretrained Model (Feature Extractor) ---
def get_model_timm(model_name: str = 'mobilenetv3_small_100.lamb_in1k'):

    model = timm.create_model(
      model_name,
      pretrained=True,
      num_classes=0,  # Para utilizar como feature extractor
    )

    return model.eval()


In [10]:
type(get_model_timm())

timm.models.mobilenetv3.MobileNetV3

In [15]:
# --- 1. Load Pretrained Model (Feature Extractor) ---
def get_model_google(model_name: str = 'google/vit-base-patch16-224'):

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = ViTModel.from_pretrained(model_name).to(device)

    return model.eval()

In [16]:
type(get_model_google())

Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


transformers.models.vit.modeling_vit.ViTModel

In [68]:
# --- 1. Load Pretrained Model (Feature Extractor) ---
def get_model_pokemon(model_name: str = 'imjeffhi/pokemon_classifier'):

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = ViTModel.from_pretrained(model_name).to(device)

    return model.eval()

In [69]:
type(get_model_pokemon())

config.json:   0%|          | 0.00/40.9k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/346M [00:00<?, ?B/s]

transformers.models.vit.modeling_vit.ViTModel

In [13]:
# --- 1. Load Pretrained Model (Feature Extractor) ---
def get_model_apple(model_name: str = 'apple/mobilevit-small'):

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = MobileViTModel.from_pretrained(model_name).to(device)

    return model.eval()

In [14]:
type(get_model_apple())

transformers.models.mobilevit.modeling_mobilevit.MobileViTModel

In [48]:
# --- 2. Compute Embedding for a Single Image ---
def get_embedding(model, img_path):

    img = Image.open(img_path)

    if img.mode != 'RGB':
      img = img.convert('RGB')


    device = "cuda" if torch.cuda.is_available() else "cpu"

    output = None

    if isinstance(model, timm.models.mobilenetv3.MobileNetV3):
        # get model specific transforms (normalization, resize)
        data_config = timm.data.resolve_model_data_config(model)
        transforms = timm.data.create_transform(**data_config, is_training=False)

        # output is (batch_size, num_features) shaped tensor
        output = model(transforms(img).unsqueeze(0))


    if isinstance(model, transformers.models.vit.modeling_vit.ViTModel):
        processor  = ViTImageProcessor.from_pretrained(model.name_or_path)
        inputs = processor(images=img, return_tensors="pt").to(device)

        last_hidden_state = model(**inputs).last_hidden_state
        output = last_hidden_state.reshape(last_hidden_state.shape[0], -1)


    if isinstance(model, transformers.models.mobilevit.modeling_mobilevit.MobileViTModel):
        processor = MobileViTImageProcessor.from_pretrained(model.name_or_path)
        inputs = processor(images=img, return_tensors="pt").to(device)

        last_hidden_state = model(**inputs).last_hidden_state
        output = last_hidden_state.reshape(last_hidden_state.shape[0], -1)


    return output #.detach().numpy()


In [50]:
get_embedding(get_model_timm(), '/content/data/database/bulbasaur/bulbasaur_1.jpeg').shape

torch.Size([1, 1024])

In [51]:
get_embedding(get_model_google(), '/content/data/database/bulbasaur/bulbasaur_1.jpeg').shape

Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


torch.Size([1, 151296])

In [52]:
get_embedding(get_model_apple(), '/content/data/database/bulbasaur/bulbasaur_1.jpeg').shape

torch.Size([1, 40960])

In [70]:
get_embedding(get_model_pokemon(), '/content/data/database/bulbasaur/bulbasaur_1.jpeg').shape

preprocessor_config.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

torch.Size([1, 151296])

In [53]:

# --- 3. Walk Dataset Folder & Compute All Embeddings ---
def compute_embeddings(
    model: torch.nn.Module,
    data_root: str | os.PathLike,
):
    """
    Compute embeddings for all images in the dataset and store them in a dictionary
    where each Pokemon has a list of embeddings.

    Args:
        model: The feature extraction model
        data_root: Root directory containing Pokemon class folders

    Returns:
        Dictionary mapping Pokemon names to lists of embeddings

    Example:
    ```
    {
        "pokemon_name": [embeddings]
    }
    ```
    """

    subfolders = [f.path for f in os.scandir(data_root) if f.is_dir()]

    embeddings = {}

    for folder in subfolders:
      label = folder.split('/')[-1]

      embeddings[label] = []

      for img_file in os.listdir(folder):
        img_path = os.path.join(folder, img_file)
        emb = get_embedding(model, img_path)

        embeddings[label].append(emb)

    return embeddings



In [54]:
#
def export_embeddings_to_pickle(embeddings, path):
    # Open the file in binary write mode ('wb')
    with open(path, 'wb') as f:
        # Dump the dictionary to the file
        pickle.dump(embeddings, f)

In [55]:
def load_embeddings_from_pickle(path):
    # Open the file in binary read mode ('rb')
    with open(path, 'rb') as f:
        # Load the dictionary from the file
        embeddings = pickle.load(f)

    return embeddings

In [56]:
# --- Cosine Similarity ---
def cosine_similarity(a, b):
    """
    Compute cosine similarity between two vectors.
    """
    return float(torch.nn.functional.cosine_similarity(a, b, dim=1))

In [110]:
# --- Single Image Similarity ---
def compute_similarity_for_image(img_path, expected_label, model, db):
    """
    Compute similarity scores between a test image and all database embeddings.
    Return

    Args:
        img_path: Path to the test image
        expected_label: Expected Pokemon label for the test image
        model: CNN/ViT model for feature extraction
        db: Database of Pokemon embeddings (loaded pickle)

    Example output:
    ```
    Expected: bulbasaur
    Top matches:
      bulbasaur    → similarity: 0.7646
      charmander   → similarity: 0.7179
      squirtle     → similarity: 0.7161
    ```
    """
    print(f"\n🔍 Testing image: {img_path}")
    # Generate embedding for the test image
    test_emb = get_embedding(model, img_path)

    # Compute similarities with all database entries
    embeddings = load_embeddings_from_pickle(db)

    similarities = []
    for label, emb_list in embeddings.items():
        similarities.append((
            label,
            max([cosine_similarity(test_emb, emb) for emb in emb_list])
        ))

    # Sort by similarity, descending
    similarities.sort(key=lambda x: x[1], reverse=True)

    # Display results
    print(f"Expected: {expected_label}")
    print("Top matches:")
    for label, similarity in similarities:
        print(f"  {label}\t→ similarity: {similarity:4f}")


In [120]:
# --- Dataset Loop ---
def compute_similarity_for_dataset(test_dir, db_path, model, compute: str = 'max_similarity'):
    """
    Compute similarity scores for all images in the test dataset.

    Args:
        test_dir: Directory containing test images organized by Pokemon
        db_path: Path to the pickle file containing database embeddings
    """

    compute = compute.lower()
    assert compute in ['max_similarity', 'vote']

    if compute == 'max_similarity':
        computing_function = compute_similarity_for_image

    if compute == 'vote':
        computing_function = compute_majority_voting_for_image

    subfolders = [f.path for f in os.scandir(test_dir) if f.is_dir()]


    for folder in subfolders:
      computing_function(
          img_path=os.path.join(folder, os.listdir(folder)[0]),
          expected_label=folder.split('/')[-1],
          model=model,
          db=db_path,
      )

In [73]:
export_embeddings_to_pickle(
    embeddings=compute_embeddings(
        model=get_model_timm(),
        data_root='/content/data/database',
    ),
    path='/content/data/embeddings/pokemon_embeddings_timm.pkl',
)

export_embeddings_to_pickle(
    embeddings=compute_embeddings(
        model=get_model_google(),
        data_root='/content/data/database',
    ),
    path='/content/data/embeddings/pokemon_embeddings_google.pkl',
)

export_embeddings_to_pickle(
    embeddings=compute_embeddings(
        model=get_model_apple(),
        data_root='/content/data/database',
    ),
    path='/content/data/embeddings/pokemon_embeddings_apple.pkl',
)

export_embeddings_to_pickle(
    embeddings=compute_embeddings(
        model=get_model_pokemon(),
        data_root='/content/data/database',
    ),
    path='/content/data/embeddings/pokemon_embeddings_pkmn.pkl',
)

In [77]:
model_params = {
    'timm': {
        'model': get_model_timm(),
        'db_path': '/content/data/embeddings/pokemon_embeddings_timm.pkl',
    },
    'google': {
        'model': get_model_google(),
        'db_path': '/content/data/embeddings/pokemon_embeddings_google.pkl',
    },
    'apple': {
        'model': get_model_apple(),
        'db_path': '/content/data/embeddings/pokemon_embeddings_apple.pkl',
    },
    'pokemon': {
        'model': get_model_pokemon(),
        'db_path': '/content/data/embeddings/pokemon_embeddings_pkmn.pkl',
    },
}

In [121]:
MODEL_TO_USE = 'timm'

compute_similarity_for_dataset(
    test_dir= '/content/data/testing',
    db_path= model_params[MODEL_TO_USE]['db_path'],
    model= model_params[MODEL_TO_USE]['model']
)


🔍 Testing image: /content/data/testing/squirtle/squirtle_1.jpeg
Expected: squirtle
Top matches:
  squirtle	→ similarity: 0.621348
  bulbasaur	→ similarity: 0.519415
  charmander	→ similarity: 0.473791

🔍 Testing image: /content/data/testing/bulbasaur/bulbasaur_1.jpeg
Expected: bulbasaur
Top matches:
  bulbasaur	→ similarity: 0.616892
  charmander	→ similarity: 0.485740
  squirtle	→ similarity: 0.474586

🔍 Testing image: /content/data/testing/charmander/charmander.jpeg
Expected: charmander
Top matches:
  bulbasaur	→ similarity: 0.626112
  squirtle	→ similarity: 0.617859
  charmander	→ similarity: 0.604219


In [122]:
MODEL_TO_USE = 'google'

compute_similarity_for_dataset(
    test_dir= '/content/data/testing',
    db_path= model_params[MODEL_TO_USE]['db_path'],
    model= model_params[MODEL_TO_USE]['model']
)


🔍 Testing image: /content/data/testing/squirtle/squirtle_1.jpeg
Expected: squirtle
Top matches:
  squirtle	→ similarity: 0.515886
  bulbasaur	→ similarity: 0.368575
  charmander	→ similarity: 0.363894

🔍 Testing image: /content/data/testing/bulbasaur/bulbasaur_1.jpeg
Expected: bulbasaur
Top matches:
  squirtle	→ similarity: 0.342315
  bulbasaur	→ similarity: 0.335128
  charmander	→ similarity: 0.276151

🔍 Testing image: /content/data/testing/charmander/charmander.jpeg
Expected: charmander
Top matches:
  charmander	→ similarity: 0.395162
  squirtle	→ similarity: 0.379982
  bulbasaur	→ similarity: 0.313849


In [123]:
MODEL_TO_USE = 'apple'

compute_similarity_for_dataset(
    test_dir= '/content/data/testing',
    db_path= model_params[MODEL_TO_USE]['db_path'],
    model= model_params[MODEL_TO_USE]['model']
)


🔍 Testing image: /content/data/testing/squirtle/squirtle_1.jpeg
Expected: squirtle
Top matches:
  squirtle	→ similarity: 0.271180
  charmander	→ similarity: 0.208435
  bulbasaur	→ similarity: 0.122585

🔍 Testing image: /content/data/testing/bulbasaur/bulbasaur_1.jpeg
Expected: bulbasaur
Top matches:
  bulbasaur	→ similarity: 0.269911
  squirtle	→ similarity: 0.265580
  charmander	→ similarity: 0.235298

🔍 Testing image: /content/data/testing/charmander/charmander.jpeg
Expected: charmander
Top matches:
  squirtle	→ similarity: 0.327032
  bulbasaur	→ similarity: 0.281768
  charmander	→ similarity: 0.280401


In [124]:
MODEL_TO_USE = 'pokemon'

compute_similarity_for_dataset(
    test_dir= '/content/data/testing',
    db_path= model_params[MODEL_TO_USE]['db_path'],
    model= model_params[MODEL_TO_USE]['model']
)


🔍 Testing image: /content/data/testing/squirtle/squirtle_1.jpeg
Expected: squirtle
Top matches:
  squirtle	→ similarity: 0.540488
  charmander	→ similarity: 0.274186
  bulbasaur	→ similarity: 0.188017

🔍 Testing image: /content/data/testing/bulbasaur/bulbasaur_1.jpeg
Expected: bulbasaur
Top matches:
  bulbasaur	→ similarity: 0.416560
  squirtle	→ similarity: 0.252849
  charmander	→ similarity: 0.133751

🔍 Testing image: /content/data/testing/charmander/charmander.jpeg
Expected: charmander
Top matches:
  charmander	→ similarity: 0.594835
  squirtle	→ similarity: 0.306891
  bulbasaur	→ similarity: 0.095034


### Compute Metrics
- Add 3 more pokemon pictures for each pokemon in the testing folder.
- Compute the classification accuracy for each Pokemon

Probamos método de voto mayoritario

In [98]:
def compute_majority_voting_for_image(img_path, expected_label, model, db):
    """
    Compute similarity scores between a test image and all database embeddings.
    Then, compute voting for the TOP 5 results.
    Return

    Args:
        img_path: Path to the test image
        expected_label: Expected Pokemon label for the test image
        model: CNN/ViT model for feature extraction
        db: Database of Pokemon embeddings (loaded pickle)

    Example output:
    ```
    Expected: bulbasaur
    Top matches:
      bulbasaur    → votes: 3
      charmander   → votes: 2
      squirtle     → votes: 1
    ```
    """
    print(f"\n🔍 Testing image: {img_path}")
    # Generate embedding for the test image
    test_emb = get_embedding(model, img_path)

    # Compute similarities with all database entries
    embeddings = load_embeddings_from_pickle(db)

    similarities = []
    for label, emb_list in embeddings.items():
        for emb in emb_list:
            similarities.append((
                label,
                cosine_similarity(test_emb, emb)
            ))

    # Sort by similarity, descending
    similarities.sort(key=lambda x: x[1], reverse=True)

    # Majority voting
    data = lambda: defaultdict(float)
    summary = defaultdict(data)
    for label, similarity in similarities[:5]:
        summary[label]['votes'] += 1
        summary[label]['max_sim'] = max(summary[label]['max_sim'], similarity)

    # Sort by votes, descending. In draw case prior max_similarity
    sorted_votes = [(label, data['votes'], data['max_sim']) for label, data in summary.items()]
    sorted_votes.sort(key=lambda x: (x[1], x[2]), reverse=True)

    # Display results showed ind ascendign order
    print(f"Expected: {expected_label}")
    print("Top matches:")
    for label, votes, max_sim in sorted_votes:
        print(f"  {label}\t→ votes: {votes}\t(max_sim: {max_sim:4f})")


In [99]:
compute_majority_voting_for_image(
    img_path='/content/data/testing/charmander/charmander.jpeg',
    expected_label='charmander',
    model=get_model_timm(),
    db='/content/data/embeddings/pokemon_embeddings_timm.pkl',
)


🔍 Testing image: /content/data/testing/charmander/charmander.jpeg
Expected: charmander
Top matches:
  squirtle	→ votes: 2.0	(max_sim: 0.617859)
  charmander	→ votes: 2.0	(max_sim: 0.604219)
  bulbasaur	→ votes: 1.0	(max_sim: 0.626112)


In [100]:
compute_majority_voting_for_image(
    img_path='/content/data/testing/charmander/charmander.jpeg',
    expected_label='charmander',
    model=get_model_pokemon(),
    db='/content/data/embeddings/pokemon_embeddings_pkmn.pkl',
)


🔍 Testing image: /content/data/testing/charmander/charmander.jpeg
Expected: charmander
Top matches:
  charmander	→ votes: 4.0	(max_sim: 0.594835)
  squirtle	→ votes: 1.0	(max_sim: 0.306891)


In [125]:
MODEL_TO_USE = 'timm'

compute_similarity_for_dataset(
    test_dir= '/content/data/testing',
    db_path= model_params[MODEL_TO_USE]['db_path'],
    model= model_params[MODEL_TO_USE]['model'],
    compute='vote'
)


🔍 Testing image: /content/data/testing/squirtle/squirtle_1.jpeg
Expected: squirtle
Top matches:
  squirtle	→ votes: 4.0	(max_sim: 0.621348)
  bulbasaur	→ votes: 1.0	(max_sim: 0.519415)

🔍 Testing image: /content/data/testing/bulbasaur/bulbasaur_1.jpeg
Expected: bulbasaur
Top matches:
  bulbasaur	→ votes: 2.0	(max_sim: 0.616892)
  charmander	→ votes: 2.0	(max_sim: 0.485740)
  squirtle	→ votes: 1.0	(max_sim: 0.474586)

🔍 Testing image: /content/data/testing/charmander/charmander.jpeg
Expected: charmander
Top matches:
  squirtle	→ votes: 2.0	(max_sim: 0.617859)
  charmander	→ votes: 2.0	(max_sim: 0.604219)
  bulbasaur	→ votes: 1.0	(max_sim: 0.626112)


In [126]:
MODEL_TO_USE = 'google'

compute_similarity_for_dataset(
    test_dir= '/content/data/testing',
    db_path= model_params[MODEL_TO_USE]['db_path'],
    model= model_params[MODEL_TO_USE]['model'],
    compute='vote'
)


🔍 Testing image: /content/data/testing/squirtle/squirtle_1.jpeg
Expected: squirtle
Top matches:
  squirtle	→ votes: 4.0	(max_sim: 0.515886)
  bulbasaur	→ votes: 1.0	(max_sim: 0.368575)

🔍 Testing image: /content/data/testing/bulbasaur/bulbasaur_1.jpeg
Expected: bulbasaur
Top matches:
  squirtle	→ votes: 3.0	(max_sim: 0.342315)
  bulbasaur	→ votes: 2.0	(max_sim: 0.335128)

🔍 Testing image: /content/data/testing/charmander/charmander.jpeg
Expected: charmander
Top matches:
  squirtle	→ votes: 3.0	(max_sim: 0.379982)
  charmander	→ votes: 2.0	(max_sim: 0.395162)


In [127]:
MODEL_TO_USE = 'apple'

compute_similarity_for_dataset(
    test_dir= '/content/data/testing',
    db_path= model_params[MODEL_TO_USE]['db_path'],
    model= model_params[MODEL_TO_USE]['model'],
    compute='vote'
)


🔍 Testing image: /content/data/testing/squirtle/squirtle_1.jpeg
Expected: squirtle
Top matches:
  squirtle	→ votes: 4.0	(max_sim: 0.271180)
  charmander	→ votes: 1.0	(max_sim: 0.208435)

🔍 Testing image: /content/data/testing/bulbasaur/bulbasaur_1.jpeg
Expected: bulbasaur
Top matches:
  bulbasaur	→ votes: 3.0	(max_sim: 0.269911)
  squirtle	→ votes: 2.0	(max_sim: 0.265580)

🔍 Testing image: /content/data/testing/charmander/charmander.jpeg
Expected: charmander
Top matches:
  squirtle	→ votes: 3.0	(max_sim: 0.327032)
  bulbasaur	→ votes: 1.0	(max_sim: 0.281768)
  charmander	→ votes: 1.0	(max_sim: 0.280401)


In [128]:
MODEL_TO_USE = 'pokemon'

compute_similarity_for_dataset(
    test_dir= '/content/data/testing',
    db_path= model_params[MODEL_TO_USE]['db_path'],
    model= model_params[MODEL_TO_USE]['model'],
    compute='vote'
)


🔍 Testing image: /content/data/testing/squirtle/squirtle_1.jpeg
Expected: squirtle
Top matches:
  squirtle	→ votes: 4.0	(max_sim: 0.540488)
  charmander	→ votes: 1.0	(max_sim: 0.274186)

🔍 Testing image: /content/data/testing/bulbasaur/bulbasaur_1.jpeg
Expected: bulbasaur
Top matches:
  bulbasaur	→ votes: 4.0	(max_sim: 0.416560)
  squirtle	→ votes: 1.0	(max_sim: 0.252849)

🔍 Testing image: /content/data/testing/charmander/charmander.jpeg
Expected: charmander
Top matches:
  charmander	→ votes: 4.0	(max_sim: 0.594835)
  squirtle	→ votes: 1.0	(max_sim: 0.306891)
